#### Expert Knowledge Worker

In [1]:
import os
import glob
from dotenv import load_dotenv
from pathlib import Path
import gradio as gr
from openai import OpenAI

In [2]:
# Setting up

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

MODEL = "gpt-4.1-nano"
openai = OpenAI()

OpenAI API Key exists and begins sk-proj-


In [39]:
import pandas as pd

knowledge = {}

# 1. Read the CSV
df = pd.read_csv("knowledge-base/smartphone-dataset.csv")

# 2. Convert to JSON (list of records) and store in the dictionary
knowledge['smartphone-dataset'] = df.to_dict(orient='records')

In [40]:
knowledge

{'smartphone-dataset': [{'brand_name': 'oneplus',
   'model': 'OnePlus Nord 6',
   'price_category': 'Premium',
   'price': 38999,
   'spec_score': 81,
   'vfm_score': 0.9068283902606832,
   'vfm_label': 'Average Value',
   'has_5G': True,
   'has_NFC': True,
   'has_IR': True,
   'processor_brand': 'qualcomm',
   'processor_name': 'Snapdragon 8s Gen4',
   'num_core': 8.0,
   'processor_speed': 3.2,
   'ram': 8.0,
   'memory': 256.0,
   'battery_capacity(mAh)': 9000.0,
   'fast_charging(W)': 80.0,
   'charging_ratio': 112.5,
   'charging_speed_type': 'Standard',
   'screen_size': 6.78,
   'refresh_rate': 165.0,
   'rear_camera': 50.0,
   'front_camera': 32.0,
   'rear_camera_count': 2,
   'os': 'Android v16'},
  {'brand_name': 'samsung',
   'model': 'Samsung Galaxy S25 Ultra',
   'price_category': 'Flagship',
   'price': 110000,
   'spec_score': 89,
   'vfm_score': 0.6585170008310294,
   'vfm_label': 'Average Value',
   'has_5G': True,
   'has_NFC': True,
   'has_IR': False,
   'proces

In [43]:
knowledge['smartphone-dataset'][0]['brand_name']

'oneplus'

In [46]:
knowledge['smartphone-dataset']

[{'brand_name': 'oneplus',
  'model': 'OnePlus Nord 6',
  'price_category': 'Premium',
  'price': 38999,
  'spec_score': 81,
  'vfm_score': 0.9068283902606832,
  'vfm_label': 'Average Value',
  'has_5G': True,
  'has_NFC': True,
  'has_IR': True,
  'processor_brand': 'qualcomm',
  'processor_name': 'Snapdragon 8s Gen4',
  'num_core': 8.0,
  'processor_speed': 3.2,
  'ram': 8.0,
  'memory': 256.0,
  'battery_capacity(mAh)': 9000.0,
  'fast_charging(W)': 80.0,
  'charging_ratio': 112.5,
  'charging_speed_type': 'Standard',
  'screen_size': 6.78,
  'refresh_rate': 165.0,
  'rear_camera': 50.0,
  'front_camera': 32.0,
  'rear_camera_count': 2,
  'os': 'Android v16'},
 {'brand_name': 'samsung',
  'model': 'Samsung Galaxy S25 Ultra',
  'price_category': 'Flagship',
  'price': 110000,
  'spec_score': 89,
  'vfm_score': 0.6585170008310294,
  'vfm_label': 'Average Value',
  'has_5G': True,
  'has_NFC': True,
  'has_IR': False,
  'processor_brand': 'qualcomm',
  'processor_name': 'Snapdragon 8 E

In [47]:
SYSTEM_PREFIX = """
You represent SmartPhone LLM, the SmartPhone Tech company.
You are an expert in answering questions about SmartPhone LLM; its brands and its models.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [57]:
def get_relevant_context(message):
    clean_message = ''.join(ch for ch in message if ch.isalnum() or ch.isspace()).lower()
    
    # This is now a list of dicts
    dataset = knowledge.get('smartphone-dataset')
    
    # Use 'not dataset' to check if a list is empty
    if not dataset:
        return []

    search_columns = ['brand_name', 'model', 'price', 'price_category', 'ram', 'memory', 'rear_camera']
    words = clean_message.split()
    if not words:
        return []

    pattern = '|'.join([rf'\b{word}\b' for word in words])
    
    # Since it's a list, we convert it back to a temporary DF to use the fast regex search
    temp_df = pd.DataFrame(dataset)
    
    combined_series = temp_df[search_columns].fillna('').astype(str).agg(' '.join, axis=1).str.lower()
    results_df = temp_df[combined_series.str.contains(pattern, regex=True)]

    return results_df.to_dict(orient='records')

In [59]:
get_relevant_context('What are the list of model OnePlus products available in knowledge base')

[{'brand_name': 'oneplus',
  'model': 'OnePlus Nord 6',
  'price_category': 'Premium',
  'price': 38999,
  'spec_score': 81,
  'vfm_score': 0.9068283902606832,
  'vfm_label': 'Average Value',
  'has_5G': True,
  'has_NFC': True,
  'has_IR': True,
  'processor_brand': 'qualcomm',
  'processor_name': 'Snapdragon 8s Gen4',
  'num_core': 8.0,
  'processor_speed': 3.2,
  'ram': 8.0,
  'memory': 256.0,
  'battery_capacity(mAh)': 9000.0,
  'fast_charging(W)': 80.0,
  'charging_ratio': 112.5,
  'charging_speed_type': 'Standard',
  'screen_size': 6.78,
  'refresh_rate': 165.0,
  'rear_camera': 50.0,
  'front_camera': 32.0,
  'rear_camera_count': 2,
  'os': 'Android v16'},
 {'brand_name': 'oneplus',
  'model': 'OnePlus Nord CE 6 5G',
  'price_category': 'Mid-Range',
  'price': 26990,
  'spec_score': 72,
  'vfm_score': 1.5223890285696045,
  'vfm_label': 'Value King',
  'has_5G': True,
  'has_NFC': False,
  'has_IR': True,
  'processor_brand': 'qualcomm',
  'processor_name': 'Octa Core Processor',

In [62]:
get_relevant_context('What are the list of products for Flagship price category in knowledge base')

[{'brand_name': 'samsung',
  'model': 'Samsung Galaxy S25 Ultra',
  'price_category': 'Flagship',
  'price': 110000,
  'spec_score': 89,
  'vfm_score': 0.6585170008310294,
  'vfm_label': 'Average Value',
  'has_5G': True,
  'has_NFC': True,
  'has_IR': False,
  'processor_brand': 'qualcomm',
  'processor_name': 'Snapdragon 8 Elite for Galaxy',
  'num_core': 8.0,
  'processor_speed': 4.47,
  'ram': 12.0,
  'memory': 256.0,
  'battery_capacity(mAh)': 5000.0,
  'fast_charging(W)': 45.0,
  'charging_ratio': 111.11111111111111,
  'charging_speed_type': 'Standard',
  'screen_size': 6.9,
  'refresh_rate': 120.0,
  'rear_camera': 200.0,
  'front_camera': 12.0,
  'rear_camera_count': 4,
  'os': 'Android v15'},
 {'brand_name': 'apple',
  'model': 'Apple iPhone 17',
  'price_category': 'Flagship',
  'price': 82900,
  'spec_score': 76,
  'vfm_score': -0.8932438636576615,
  'vfm_label': 'Average Value',
  'has_5G': True,
  'has_NFC': True,
  'has_IR': False,
  'processor_brand': 'apple',
  'process

In [65]:
def additional_context(message):
    relevant_context = get_relevant_context(message)
    
    if not relevant_context:
        return "There is no additional context relevant to the user's question."
    
    result = "The following additional context might be relevant in answering the user's question:\n\n"
    
    # Refactored: Convert each dictionary row into a formatted string
    context_strings = []
    for row in relevant_context:
        # Create a string representation of the row (e.g., "brand_name: Motorola, model: Edge 40...")
        row_str = ", ".join([f"{key}: {val}" for key, val in row.items()])
        context_strings.append(row_str)
    
    # Now .join() will work because context_strings contains only strings
    result += "\n\n".join(context_strings)
    
    return result

In [66]:
print(additional_context("What are the motorola smartphones available in knowledge base"))

The following additional context might be relevant in answering the user's question:

brand_name: motorola, model: Motorola Edge 70 Fusion, price_category: Mid-Range, price: 25999, spec_score: 71, vfm_score: 1.34842668766501, vfm_label: Value King, has_5G: True, has_NFC: True, has_IR: False, processor_brand: qualcomm, processor_name: Snapdragon 7s Gen4, num_core: 8.0, processor_speed: 2.7, ram: 8.0, memory: 128.0, battery_capacity(mAh): 7000.0, fast_charging(W): 68.0, charging_ratio: 102.94117647058825, charging_speed_type: Standard, screen_size: 6.78, refresh_rate: 144.0, rear_camera: 50.0, front_camera: 32.0, rear_camera_count: 3, os: Android v16

brand_name: motorola, model: Motorola Edge 60 Fusion, price_category: Mid-Range, price: 20900, spec_score: 67, vfm_score: 0.6525773240466323, vfm_label: Average Value, has_5G: True, has_NFC: False, has_IR: False, processor_brand: mediatek, processor_name: Dimensity 7400, num_core: 8.0, processor_speed: 2.5, ram: 8.0, memory: 256.0, battery_

In [67]:
def chat(message, history):
    system_message = SYSTEM_PREFIX + additional_context(message)
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

In [68]:
view = gr.ChatInterface(chat, type="messages").launch(inbrowser=True)


* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
